# Pair pool and fixed-scale z-score training: local V2

Screen equities, commodity ETF/ETPs and other ETFs. Use the original `TS_implement.ipynb` / `TS_lib.py` **mean_square** scale and single-Z strategy. There is no rolling window in this version. Only `data/pair_pool.csv` is saved; no price cache or intermediate CSV is written. The old A/B/C version is preserved in GitHub tag `v1.0.0`; this V2 branch stays local.

In [1]:
from pathlib import Path
import logging
import pandas as pd
from IPython.display import display
from pair_pool import PairPoolConfig
from pair_training import SignalTrainingConfig, PairSignalTrainer

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

In [2]:
pool_config = PairPoolConfig(
    target_pairs=300,
    target_commodity_pairs=50,
    target_other_etf_pairs=50,
    min_market_cap=10_000_000_000,
    min_etf_assets=100_000_000,
    min_average_dollar_volume=20_000_000,
    output_file=Path("data/pair_pool.csv"),
)
training_config = SignalTrainingConfig(
    train_start="2022-01-01",
    train_end="2024-12-31",
    validation_start="2025-01-01",
    validation_end="2025-12-31",
    z_values=tuple(i / 4 for i in range(1, 15)),  # 0.25, 0.50, ..., 3.50
    exit_on_cross_mean=False,                    # Exit at Z / 2
    prevent_immediate_reentry=True,
    stop_loss=0.12,
    min_train_trades=5,
    min_validation_trades=2,
    long_amount=100_000,
)

## Method and assumptions

- **2022-2024:** screen pairs, fit Close OLS and residual AR(1), and estimate fixed `mu_e`, `theta` and `sigma_eq` with the original `mean_square` formula. There is no rolling window. Training performance is in-sample.
- Use `z=(spread-mu_e)/sigma_eq`. Test Z from 0.25 to 3.50; default exit is Z/2. Choose the highest training budget return, then Sharpe, with at least five completed trades.
- **2025:** validate once from flat positions with all training estimates and Z frozen. Require at least two validation trades and positive training/validation P&L and Sharpe. Keep failed rows with reasons. No 2026 prices are requested or used. Separate `refit_*` fields use data through 2025 for future deployment only.
- Close signals fill at the next common trading day's Open. High z shorts Y/longs X; low z reverses it. Exit on returning to or crossing the exit threshold. The 12% trade stop also fills next Open; gaps can exceed it. Suppress entry signals on a day that just executed an exit.
- Each pair has a fixed USD 200,000 budget, entry caps of USD 100,000 per leg, and integer shares held until exit. Returns equal P&L divided by that fixed budget. Flatten at each period's last Close; record `forced_exits` and `stop_exits`. Fees, borrow, financing and distribution cash flows remain zero.
- These are independent pair results, not one combined portfolio. Existing liquidity/EG/ADF filters remain; current metadata still cause survivorship/selection bias. Full formulas and accounting details are in [README.md](README.md).

In [3]:
trainer = PairSignalTrainer(training_config, pool_config=pool_config)
pair_pool = trainer.run()

display(pair_pool.groupby("pair_family").agg(
    candidates=("pair_id", "size"),
    validation_passed=("validation_passed", "sum"),
))
print("Saved:", pool_config.output_file)
print("Parameter training: 2022-2024 | Validation: 2025 | No 2026 prices")

INFO: Industry screens 10/145; 67 equity records


INFO: Industry screens 20/145; 177 equity records


INFO: Industry screens 30/145; 226 equity records


INFO: Industry screens 40/145; 290 equity records


INFO: Industry screens 50/145; 344 equity records


...

INFO: Signal training 300/333; validation passed=142


INFO: Signal training 325/333; validation passed=160


INFO: Signal training 333/333; validation passed=162


,candidates,validation_passed
pair_family,,
commodity_etf,6,1
equity,277,128
other_etf,50,33


Saved: data\pair_pool.csv
Parameter training: 2022-2024 | Validation: 2025 | No 2026 prices


In [4]:
columns = [
    "pair_id", "pair_family", "entry_z", "exit_z", "train_mu_e", "train_sigma_eq",
    "train_trades", "train_return_on_budget",
    "validation_trades", "validation_pnl", "validation_return_on_budget",
    "validation_sharpe", "validation_max_drawdown", "validation_passed", "validation_status",
]
display(pair_pool[columns].head(25).round(4))
ready_pairs = pair_pool.loc[pair_pool["validation_passed"]]
print("Pairs passing the validation rule:", len(ready_pairs))
display(ready_pairs[columns].head(20).round(4))
display(pair_pool["validation_status"].value_counts().to_frame("pairs"))

,pair_id,pair_family,entry_z,exit_z,train_mu_e,train_sigma_eq,train_trades,train_return_on_budget,validation_trades,validation_pnl,validation_return_on_budget,validation_sharpe,validation_max_drawdown,validation_passed,validation_status
0,TSCO__DKS,equity,1.75,0.875,-0.2034,0.4858,43,0.7272,23,79925.9332,0.3996,3.1688,-0.0493,True,passed
1,IOT__FICO,equity,0.75,0.375,-0.5302,0.8207,48,1.7291,19,123097.5310,0.6155,2.4477,-0.1196,True,passed
2,IJH__MDY,other_etf,2.25,1.125,-0.0001,0.0070,61,0.0198,20,2541.3909,0.0127,2.4376,-0.0010,True,passed
3,APO__AMP,equity,0.25,0.125,0.1735,1.4793,54,0.5949,16,43679.2039,0.2184,2.3767,-0.0292,True,passed
4,PSA__UDR,equity,2.75,1.375,0.6915,3.5756,29,0.5298,9,22666.1683,0.1133,2.2562,-0.0445,True,passed
5,CW__EMBJ,equity,1.50,0.750,3.0341,4.8012,24,0.7974,18,77546.1149,0.3877,2.1302,-0.1561,True,passed
6,IVE__IUSV,other_etf,1.25,0.625,0.0043,0.0592,49,0.0229,20,2333.7848,0.0117,1.9745,-0.0018,True,passed
7,VTI__ITOT,other_etf,0.50,0.250,-0.0017,0.0220,69,0.0161,26,1947.7948,0.0097,1.9269,-0.0019,True,passed
8,VICI__GLPI,equity,0.75,0.375,-0.0129,0.1668,51,0.3436,8,20477.6782,0.1024,1.8419,-0.0478,True,passed
9,SAP__CHKP,equity,1.00,0.500,-1.0290,4.4477,27,0.5550,11,45944.1614,0.2297,1.8199,-0.0449,True,passed


Pairs passing the validation rule: 162


,pair_id,pair_family,entry_z,exit_z,train_mu_e,train_sigma_eq,train_trades,train_return_on_budget,validation_trades,validation_pnl,validation_return_on_budget,validation_sharpe,validation_max_drawdown,validation_passed,validation_status
0,TSCO__DKS,equity,1.75,0.875,-0.2034,0.4858,43,0.7272,23,79925.9332,0.3996,3.1688,-0.0493,True,passed
1,IOT__FICO,equity,0.75,0.375,-0.5302,0.8207,48,1.7291,19,123097.5310,0.6155,2.4477,-0.1196,True,passed
2,IJH__MDY,other_etf,2.25,1.125,-0.0001,0.0070,61,0.0198,20,2541.3909,0.0127,2.4376,-0.0010,True,passed
3,APO__AMP,equity,0.25,0.125,0.1735,1.4793,54,0.5949,16,43679.2039,0.2184,2.3767,-0.0292,True,passed
4,PSA__UDR,equity,2.75,1.375,0.6915,3.5756,29,0.5298,9,22666.1683,0.1133,2.2562,-0.0445,True,passed
5,CW__EMBJ,equity,1.50,0.750,3.0341,4.8012,24,0.7974,18,77546.1149,0.3877,2.1302,-0.1561,True,passed
6,IVE__IUSV,other_etf,1.25,0.625,0.0043,0.0592,49,0.0229,20,2333.7848,0.0117,1.9745,-0.0018,True,passed
7,VTI__ITOT,other_etf,0.50,0.250,-0.0017,0.0220,69,0.0161,26,1947.7948,0.0097,1.9269,-0.0019,True,passed
8,VICI__GLPI,equity,0.75,0.375,-0.0129,0.1668,51,0.3436,8,20477.6782,0.1024,1.8419,-0.0478,True,passed
9,SAP__CHKP,equity,1.00,0.500,-1.0290,4.4477,27,0.5550,11,45944.1614,0.2297,1.8199,-0.0449,True,passed


,pairs
validation_status,
passed,162
validation_not_profitable,158
too_few_validation_trades,9
data_or_model_error,3
training_not_profitable,1


## ETF coverage

The catalog covers bullion, energy, agriculture, industrial metals, broad commodities, equity indices/sectors and bonds. A category can have zero accepted pairs. Same-index/physical-metal `tracking_pair` candidates are explicitly flagged; they do not establish a trading edge or tax eligibility. Selection targets are not filled by weakening thresholds.

In [5]:
with pd.option_context("display.max_rows", None):
    display(trainer.pool_result["etf_coverage"])
print("Source events:", trainer.pool_result["events"][:10])

,pair_family,exposure_group,catalog_funds,metadata_funds,size_eligible_funds,formation_eligible_funds,accepted_pairs_involving_group
0,commodity_etf,agriculture_basket_futures,3,3,2,0,0
1,commodity_etf,battery_metals_futures,1,1,0,0,0
2,commodity_etf,broad_commodity_futures,12,12,12,1,0
3,commodity_etf,copper_futures,1,1,1,0,0
4,commodity_etf,corn_futures,1,1,1,0,0
5,commodity_etf,crude_oil_futures,4,4,3,0,0
6,commodity_etf,energy_basket_futures,1,1,1,0,0
7,commodity_etf,gasoline_futures,1,1,1,0,0
8,commodity_etf,gold_physical,8,8,8,8,5
9,commodity_etf,industrial_metals_futures,1,1,1,0,0


Source events: ['missing_history SPCX: No closing prices', 'missing_history SKHY: No closing prices', 'missing_history SNDK: No closing prices', 'missing_history HONA: No closing prices', 'missing_history VYLR: No closing prices', 'missing_history MDLN: No closing prices', 'missing_history SUNB: No closing prices', 'missing_history VG: No closing prices', 'missing_history CRWV: No closing prices', 'missing_history BSP: No closing prices']
